# Chapter 24: Factor Graphs

<a href="../lite/lab/index.html?path=ch24_factor_graphs.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse, FancyArrowPatch

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

def draw_cov_ellipse(ax, mean, cov, n_std=2, **kwargs):
    from matplotlib.patches import Ellipse
    vals, vecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(vecs[1,1], vecs[0,1]))
    w, h = 2 * n_std * np.sqrt(np.maximum(vals, 0))
    ax.add_patch(Ellipse(xy=mean, width=w, height=h, angle=angle, **kwargs))

Draw a dot for every robot pose. Draw a dot for every landmark. Now draw a line
between a pose and a landmark whenever the robot saw that landmark from that
pose. Draw lines between consecutive poses for odometry. You just drew a factor
graph, the data structure that makes modern SLAM fast.

This chapter introduces **factor graphs** as the unifying representation for
SLAM: **variable nodes**, **factor nodes**, the **bipartite structure**, and
the critical role of **sparsity** in making optimization efficient.

```{admonition} What you will build
:class: tip

- Draw a factor graph for a SLAM problem with poses, landmarks, and observations
- Construct the Jacobian and information matrix and visualize their sparsity patterns
- Understand why sparsity makes large SLAM problems tractable
- See how adding a loop closure creates new connections in the graph

**Real world application:** Factor graphs are the data structure behind GTSAM, iSAM, and most modern SLAM frameworks. After this chapter, you will be able to read and construct the factor graphs used in SLAM papers.
```

```{admonition} Libraries and tools used in practice
:class: note

In this chapter we implement everything from scratch for learning. In production, engineers use these libraries:

| Library / Tool | What it does |
|---|---|
| **GTSAM** | The most popular factor graph library. Supports incremental solving (iSAM2) |
| **g2o** | Factor graph optimization used in ORB-SLAM, RTAB-Map, and many others |
| **Ceres Solver** | General purpose but can model factor graphs via residual blocks |

Implementing from scratch teaches you **why** these tools work. Using them in production saves you from reinventing the wheel.
```

## 24.1 Variables and Factors

A **factor graph** is a bipartite graph with two types of nodes:

- **Variable nodes** (circles): the unknowns we want to estimate. In SLAM,
  these are robot poses $\mathbf{x}_i$ and landmark positions $\mathbf{l}_j$.

- **Factor nodes** (squares): each factor encodes one constraint (measurement
  or prior). It connects to the variables involved in that constraint.

The joint probability factorizes as:

$$p(\mathbf{X}) \propto \prod_k f_k(\mathbf{X}_k)$$

where $\mathbf{X}_k$ is the subset of variables connected to factor $k$.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
n_poses = 4
n_landmarks = 3
# Observations: (pose_idx, landmark_idx)
observations = [(0, 0), (1, 0), (1, 1), (2, 1), (2, 2), (3, 2), (3, 0)]
# ──────────────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(12, 7))

# Position variables
pose_y = 3.0
lm_y = 7.0
pose_xs = np.linspace(1, 10, n_poses)
lm_xs = np.linspace(2, 9, n_landmarks)

# Draw variable nodes (circles)
for i, px in enumerate(pose_xs):
    ax.plot(px, pose_y, 'o', color='steelblue', ms=25, zorder=5)
    ax.text(px, pose_y, f'$x_{i}$', ha='center', va='center',
            fontsize=12, fontweight='bold', color='white', zorder=6)

for j, lx in enumerate(lm_xs):
    ax.plot(lx, lm_y, 'o', color='forestgreen', ms=25, zorder=5)
    ax.text(lx, lm_y, f'$l_{j}$', ha='center', va='center',
            fontsize=12, fontweight='bold', color='white', zorder=6)

# Draw odometry factors (between consecutive poses)
factor_idx = 0
for i in range(n_poses - 1):
    mid_x = (pose_xs[i] + pose_xs[i+1]) / 2
    mid_y = pose_y - 1.0
    ax.plot(mid_x, mid_y, 's', color='orange', ms=15, zorder=5)
    ax.text(mid_x, mid_y, f'$f_{{{factor_idx}}}$', ha='center', va='center',
            fontsize=8, fontweight='bold', zorder=6)
    ax.plot([pose_xs[i], mid_x], [pose_y, mid_y], 'gray', lw=1.5)
    ax.plot([pose_xs[i+1], mid_x], [pose_y, mid_y], 'gray', lw=1.5)
    factor_idx += 1

# Draw observation factors
for pi, lj in observations:
    mid_x = (pose_xs[pi] + lm_xs[lj]) / 2
    mid_y = (pose_y + lm_y) / 2
    ax.plot(mid_x, mid_y, 's', color='tomato', ms=12, zorder=5)
    ax.text(mid_x, mid_y, f'$f_{{{factor_idx}}}$', ha='center', va='center',
            fontsize=7, fontweight='bold', zorder=6)
    ax.plot([pose_xs[pi], mid_x], [pose_y, mid_y], 'gray', lw=1, alpha=0.6)
    ax.plot([lm_xs[lj], mid_x], [lm_y, mid_y], 'gray', lw=1, alpha=0.6)
    factor_idx += 1

# Prior factor on x0
ax.plot(pose_xs[0] - 0.8, pose_y, 's', color='purple', ms=12, zorder=5)
ax.plot([pose_xs[0] - 0.8, pose_xs[0]], [pose_y, pose_y], 'gray', lw=1.5)
ax.text(pose_xs[0] - 0.8, pose_y - 0.5, 'prior', ha='center', fontsize=9, color='purple')

# Legend
ax.plot([], [], 'o', color='steelblue', ms=12, label='Pose variables')
ax.plot([], [], 'o', color='forestgreen', ms=12, label='Landmark variables')
ax.plot([], [], 's', color='orange', ms=10, label='Odometry factors')
ax.plot([], [], 's', color='tomato', ms=10, label='Observation factors')
ax.plot([], [], 's', color='purple', ms=10, label='Prior factor')

ax.set_xlim(-1, 12); ax.set_ylim(0.5, 9)
ax.set_title('Factor graph for SLAM', fontsize=14)
ax.legend(loc='upper right', fontsize=10)
ax.axis('off')

plt.tight_layout()
plt.show()

print(f'Variables: {n_poses} poses + {n_landmarks} landmarks = {n_poses + n_landmarks}')
print(f'Factors: 1 prior + {n_poses-1} odometry + {len(observations)} observations = {1 + n_poses - 1 + len(observations)}')

**Key insight:** Every measurement creates one factor. Each factor only
connects to the variables it involves. An odometry factor connects two
consecutive poses. An observation factor connects one pose and one landmark.
Most variables are **not** directly connected to most other variables.

## 24.2 Graph Structure: Bipartite Graph

A factor graph is **bipartite**: edges only go between variable nodes and
factor nodes, never between two variables or two factors.

The adjacency structure tells us:
- Which variables does each factor constrain?
- Which factors constrain each variable?

This can be represented as an **adjacency matrix** or, equivalently, as the
**structure of the Jacobian** matrix.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
# Using the same graph from above
# Each variable: pose has 3 DOF (x,y,theta), landmark has 2 DOF (x,y)
pose_dim = 3
lm_dim = 2
# ──────────────────────────────────────────────────────────────────────────────

n_var = n_poses * pose_dim + n_landmarks * lm_dim

# Build Jacobian structure (which entries are nonzero)
# Factors:
# Prior: 1 factor, 3 residuals, connects to x0
# Odometry: n_poses-1 factors, 3 residuals each, connects to x_i and x_{i+1}
# Observations: len(observations) factors, 2 residuals each, connects to x_i and l_j

n_residuals = 3 + (n_poses - 1) * 3 + len(observations) * 2
J_structure = np.zeros((n_residuals, n_var))

row = 0
# Prior on x0
J_structure[row:row+3, 0:3] = 1
row += 3

# Odometry
for i in range(n_poses - 1):
    col_i = i * pose_dim
    col_j = (i+1) * pose_dim
    J_structure[row:row+3, col_i:col_i+3] = 1
    J_structure[row:row+3, col_j:col_j+3] = 1
    row += 3

# Observations
for pi, lj in observations:
    col_pose = pi * pose_dim
    col_lm = n_poses * pose_dim + lj * lm_dim
    J_structure[row:row+2, col_pose:col_pose+3] = 1
    J_structure[row:row+2, col_lm:col_lm+2] = 1
    row += 2

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.spy(J_structure, aspect='auto', markersize=4, color='steelblue')
ax.set_xlabel('Variable index', fontsize=12)
ax.set_ylabel('Residual (factor) index', fontsize=12)
ax.set_title(f'Jacobian structure: {n_residuals}×{n_var}', fontsize=13)

# Add variable labels
x_labels = []
for i in range(n_poses):
    x_labels.extend([f'x{i}x', f'x{i}y', f'x{i}θ'])
for j in range(n_landmarks):
    x_labels.extend([f'l{j}x', f'l{j}y'])
ax.set_xticks(range(n_var))
ax.set_xticklabels(x_labels, fontsize=6, rotation=90)

# Information matrix H = J^T J
H = J_structure.T @ J_structure

ax = axes[1]
ax.spy(H, aspect='auto', markersize=6, color='tomato')
ax.set_title(f'Information matrix $J^T J$: {n_var}×{n_var}', fontsize=13)
ax.set_xticks(range(n_var))
ax.set_xticklabels(x_labels, fontsize=6, rotation=90)
ax.set_yticks(range(n_var))
ax.set_yticklabels(x_labels, fontsize=6)

plt.tight_layout()
plt.show()

total = n_var * n_var
nonzero = np.count_nonzero(H)
print(f'Information matrix: {n_var}×{n_var} = {total} entries')
print(f'Nonzero entries: {nonzero} ({100*nonzero/total:.1f}%)')
print(f'Zero entries: {total - nonzero} ({100*(total-nonzero)/total:.1f}%)')

**Observation:** The Jacobian is very sparse. Most entries are zero because
most factors only touch a few variables. The information matrix $J^T J$ inherits
this sparsity. This is the foundation of efficient SLAM optimization.

## 24.3 Sparsity: Why It Matters

The Gauss-Newton update requires solving:

$$(J^T \Omega \, J) \, \Delta\mathbf{x} = -J^T \Omega \, \mathbf{r}$$

The matrix $H = J^T \Omega \, J$ is the **information matrix** (or
**Hessian approximation**). If $H$ were dense, solving this system would cost
$O(n^3)$. But because $H$ is **sparse**, we can use sparse solvers
(Cholesky factorization with fill-reducing ordering) that exploit the
zero structure.

The sparsity pattern comes directly from the factor graph:
- $H_{ij} \neq 0$ if and only if variables $i$ and $j$ share at least one factor
- Odometry creates a **banded** structure (tridiagonal for poses)
- Observations create connections between poses and landmarks
- Loop closures create **fill-in** (new nonzero entries far from the diagonal)

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
n_poses_big = 20
n_lm_big = 6
obs_big = [(0,0), (1,0), (2,1), (3,1), (5,2), (6,2), (8,3), (9,3),
           (11,4), (12,4), (14,5), (15,5), (18,0), (19,0)]  # last two = loop closure
# ──────────────────────────────────────────────────────────────────────────────

def build_info_matrix(n_poses, n_lm, observations, pose_d=3, lm_d=2):
    """Build information matrix sparsity pattern from factor graph."""
    n = n_poses * pose_d + n_lm * lm_d
    H = np.zeros((n, n))
    
    # Prior on first pose
    H[:pose_d, :pose_d] += np.eye(pose_d)
    
    # Odometry
    for i in range(n_poses - 1):
        ci = i * pose_d
        cj = (i+1) * pose_d
        # Diagonal blocks
        H[ci:ci+pose_d, ci:ci+pose_d] += np.eye(pose_d)
        H[cj:cj+pose_d, cj:cj+pose_d] += np.eye(pose_d)
        # Off-diagonal blocks
        H[ci:ci+pose_d, cj:cj+pose_d] += np.eye(pose_d) * 0.5
        H[cj:cj+pose_d, ci:ci+pose_d] += np.eye(pose_d) * 0.5
    
    # Observations
    for pi, lj in observations:
        cp = pi * pose_d
        cl = n_poses * pose_d + lj * lm_d
        H[cp:cp+pose_d, cp:cp+pose_d] += np.eye(pose_d) * 0.3
        H[cl:cl+lm_d, cl:cl+lm_d] += np.eye(lm_d) * 0.3
        H[cp:cp+min(pose_d,lm_d), cl:cl+lm_d] += np.eye(lm_d) * 0.2
        H[cl:cl+lm_d, cp:cp+min(pose_d,lm_d)] += np.eye(lm_d) * 0.2
    
    return H

# Without loop closure
obs_no_loop = obs_big[:-2]
H_no_loop = build_info_matrix(n_poses_big, n_lm_big, obs_no_loop)

# With loop closure
H_with_loop = build_info_matrix(n_poses_big, n_lm_big, obs_big)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.spy(H_no_loop, aspect='auto', markersize=2, color='steelblue')
ax.set_title('Information matrix: NO loop closure', fontsize=13)
nonz_no = np.count_nonzero(H_no_loop)
total_big = H_no_loop.shape[0]**2
ax.set_xlabel(f'{nonz_no} nonzeros ({100*nonz_no/total_big:.1f}%)')

ax = axes[1]
ax.spy(H_with_loop, aspect='auto', markersize=2, color='tomato')
ax.set_title('Information matrix: WITH loop closure', fontsize=13)
nonz_lc = np.count_nonzero(H_with_loop)
ax.set_xlabel(f'{nonz_lc} nonzeros ({100*nonz_lc/total_big:.1f}%)')

# Highlight the loop closure entry
lc_pose = 18 * 3
lc_lm = n_poses_big * 3 + 0 * 2
ax.plot(lc_lm, lc_pose, 'o', color='orange', ms=10, zorder=10)
ax.annotate('loop closure!', xy=(lc_lm, lc_pose), xytext=(lc_lm+5, lc_pose-5),
            arrowprops=dict(arrowstyle='->', color='orange', lw=2),
            fontsize=11, color='orange', fontweight='bold')

plt.tight_layout()
plt.show()

**Key insight:** Loop closure adds connections between distant poses and
landmarks, creating new nonzero entries far from the diagonal. These entries
are exactly what propagates corrections across the trajectory during
optimization. More fill-in means more work for the solver, but also more
information linking distant parts of the map.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
sizes = [10, 20, 50, 100, 200]
obs_density = 0.3  # fraction of pose-landmark pairs observed
# ──────────────────────────────────────────────────────────────────────────────

sparsity_ratios = []
dense_ops = []
sparse_ops_est = []

np.random.seed(42)
for n_p in sizes:
    n_l = max(3, n_p // 5)
    n_total = n_p * 3 + n_l * 2
    
    # Generate random observations
    obs_list = []
    for i in range(n_p):
        for j in range(n_l):
            if np.random.rand() < obs_density / n_l:
                obs_list.append((i, j))
    
    H_test = build_info_matrix(n_p, n_l, obs_list)
    nnz = np.count_nonzero(H_test)
    sparsity_ratios.append(nnz / n_total**2)
    dense_ops.append(n_total**3)  # dense Cholesky
    sparse_ops_est.append(nnz * n_total)  # rough sparse estimate

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(sizes, [s*100 for s in sparsity_ratios], 'steelblue', lw=2, marker='o')
ax.set_xlabel('Number of poses', fontsize=12)
ax.set_ylabel('Fraction nonzero (%)', fontsize=12)
ax.set_title('Sparsity: fraction of nonzero entries in H', fontsize=13)

ax = axes[1]
ax.loglog(sizes, dense_ops, 'tomato', lw=2, marker='s', label='Dense $O(n^3)$')
ax.loglog(sizes, sparse_ops_est, 'forestgreen', lw=2, marker='^', label='Sparse (estimated)')
ax.set_xlabel('Number of poses', fontsize=12)
ax.set_ylabel('Operations', fontsize=12)
ax.set_title('Solving cost: dense vs sparse', fontsize=13)
ax.legend()

plt.tight_layout()
plt.show()

print('As the problem grows, sparsity increases and sparse solvers win by huge margins.')

**Observation:** For realistic SLAM problems, the information matrix is
overwhelmingly sparse (often >95% zeros). Sparse solvers exploit this to
solve problems with thousands of variables in milliseconds, something that
would take minutes with a dense solver.

---

## Capstone: Building and Analyzing a Factor Graph

A robot drives a loop past 6 landmarks. We construct the full factor graph,
build the Jacobian and information matrix, visualize the sparsity pattern,
and show how loop closure changes the structure.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
np.random.seed(42)
n_poses_cap = 12
n_lm_cap = 6
# Robot drives a rectangle, observing nearby landmarks
pose_positions = []
for i in range(3): pose_positions.append([i*2, 0])
for i in range(3): pose_positions.append([4, i*2])
for i in range(3): pose_positions.append([4-i*2, 4])
for i in range(3): pose_positions.append([0, 4-i*2])
pose_positions = np.array(pose_positions, dtype=float)

lm_positions = np.array([[1, 2], [3, 1], [5, 3], [3, 5], [1, 5], [-1, 3]], dtype=float)
obs_range = 3.5
# ──────────────────────────────────────────────────────────────────────────────

# Determine which landmarks each pose can see
obs_cap = []
for i, pp in enumerate(pose_positions):
    for j, lp in enumerate(lm_positions):
        if np.linalg.norm(pp - lp) < obs_range:
            obs_cap.append((i, j))

# Build the factor graph visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: spatial layout with connections
ax = axes[0]
ax.plot(pose_positions[:, 0], pose_positions[:, 1], 'steelblue', lw=2, marker='o',
        ms=10, zorder=5, label='Robot poses')
ax.scatter(lm_positions[:, 0], lm_positions[:, 1], c='forestgreen', s=120,
           marker='*', zorder=5, label='Landmarks')

# Draw observation connections
for pi, lj in obs_cap:
    ax.plot([pose_positions[pi, 0], lm_positions[lj, 0]],
            [pose_positions[pi, 1], lm_positions[lj, 1]],
            'tomato', lw=0.8, alpha=0.4)

for i, pp in enumerate(pose_positions):
    ax.annotate(f'$x_{{{i}}}$', xy=pp, xytext=(5, -12), textcoords='offset points',
                fontsize=9, color='steelblue')
for j, lp in enumerate(lm_positions):
    ax.annotate(f'$l_{j}$', xy=lp, xytext=(5, 5), textcoords='offset points',
                fontsize=10, color='forestgreen', fontweight='bold')

ax.set_aspect('equal')
ax.set_title(f'Spatial layout: {len(obs_cap)} observations', fontsize=13)
ax.legend(loc='lower right')

# Right: information matrix
H_cap = build_info_matrix(n_poses_cap, n_lm_cap, obs_cap)

ax = axes[1]
ax.spy(H_cap, aspect='auto', markersize=3, color='steelblue')
n_total_cap = H_cap.shape[0]
nnz_cap = np.count_nonzero(H_cap)
ax.set_title(f'Information matrix: {n_total_cap}x{n_total_cap}, '
             f'{nnz_cap} nonzeros ({100*nnz_cap/n_total_cap**2:.1f}%)', fontsize=12)

# Draw separator between poses and landmarks
sep = n_poses_cap * 3
ax.axhline(sep - 0.5, color='orange', lw=1, ls='--')
ax.axvline(sep - 0.5, color='orange', lw=1, ls='--')
ax.text(sep/2, -2.5, 'Poses', ha='center', fontsize=10, color='steelblue')
ax.text(sep + (n_total_cap-sep)/2, -2.5, 'LMs', ha='center', fontsize=10, color='forestgreen')

plt.tight_layout()
plt.show()

print(f'Variables: {n_poses_cap} poses (3 DOF each) + {n_lm_cap} landmarks (2 DOF each) = {n_total_cap} DOF')
print(f'Factors: 1 prior + {n_poses_cap-1} odometry + {len(obs_cap)} observations = {1+n_poses_cap-1+len(obs_cap)}')

In [ ]:
# Add loop closure and show the change
# Loop closure: pose 11 re-observes landmarks seen by pose 0
obs_loop = obs_cap.copy()
# Pose 11 is near pose 0, so it should see the same landmarks
for j in range(n_lm_cap):
    if np.linalg.norm(pose_positions[0] - lm_positions[j]) < obs_range:
        if (n_poses_cap - 1, j) not in obs_loop:
            obs_loop.append((n_poses_cap - 1, j))

H_loop = build_info_matrix(n_poses_cap, n_lm_cap, obs_loop)

# Show the difference
H_diff = np.abs(H_loop) - np.abs(H_cap)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

ax = axes[0]
ax.spy(H_cap, aspect='auto', markersize=3, color='steelblue')
ax.set_title('Before loop closure', fontsize=13)

ax = axes[1]
ax.spy(H_loop, aspect='auto', markersize=3, color='tomato')
ax.set_title('After loop closure', fontsize=13)

ax = axes[2]
ax.spy(H_diff, aspect='auto', markersize=5, color='orange')
ax.set_title('New nonzero entries from loop closure', fontsize=13)

plt.suptitle('Impact of loop closure on information matrix structure', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

nnz_before = np.count_nonzero(H_cap)
nnz_after = np.count_nonzero(H_loop)
nnz_new = np.count_nonzero(H_diff > 0)
print(f'Nonzeros before loop closure: {nnz_before}')
print(f'Nonzeros after loop closure:  {nnz_after}')
print(f'New nonzero entries:          {nnz_new}')
print(f'\nThese new entries link the end of the trajectory to the beginning,')
print(f'enabling the optimizer to correct drift accumulated over the entire loop.')

**Capstone observations:**
- The factor graph structure maps directly to the sparsity of the Jacobian and information matrix.
- Without loop closure, the information matrix has a banded structure (nearby poses are connected).
- Loop closure adds entries linking the last pose to the first, creating a bridge across the trajectory.
- These new entries are exactly what allows the optimizer to distribute corrections across the entire map.
- Sparse solvers exploit this structure to solve SLAM problems orders of magnitude faster than dense solvers.

---

## Exercises

### Exercise 24.1: Count the degrees of freedom

For a SLAM problem with 50 poses (3 DOF each) and 20 landmarks (2 DOF each),
compute: (a) total variable dimension, (b) information matrix size, (c) fraction
of nonzero entries if each pose observes 3 landmarks on average.

In [ ]:
# Your code here

### Exercise 24.2: Build your own factor graph

Create a factor graph for a robot that drives in a figure-8 pattern with
8 poses and 4 landmarks. Draw the graph and compute the information matrix.
How many loop closures does the figure-8 create?

In [ ]:
# Your code here

### Exercise 24.3: Fill-in analysis (challenge)

When we factorize the information matrix $H = L L^T$ (Cholesky), new
nonzero entries appear in $L$ that were zero in $H$. This is called **fill-in**.
Build a SLAM information matrix for 20 poses and 5 landmarks. Compute the
Cholesky factorization and compare the sparsity of $H$ vs $L$. How much
fill-in occurs?

In [ ]:
# Your code here
# H = build_info_matrix(...)
# Make H positive definite: H += small * I
# L = np.linalg.cholesky(H)
# Compare np.count_nonzero(H) vs np.count_nonzero(L)